<a href="https://colab.research.google.com/github/SiaXuan/CS5330_Project02/blob/main/speechbrain_vad_pruning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1：install all dependencies
!pip install torch==2.2.0 torchaudio==2.2.0 torchvision==0.17.0 numpy==1.26.4 speechbrain==1.0.0 huggingface_hub==0.23.0 --extra-index-url https://download.pytorch.org/whl/cpu -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 186.7/186.7 MB 6.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 36.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 64.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 86.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.1/760.1 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.2/788.2 kB 42.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires

In [1]:
# Cell 2：verify
import torch, torchaudio, speechbrain, numpy
print(torch.__version__)
print(torchaudio.__version__)
print(speechbrain.__version__)
print(numpy.__version__)

2.2.0+cpu
2.2.0+cpu
1.0.0
1.26.4


In [2]:
# Cell 3：Loading model
from speechbrain.inference.VAD import VAD

vad = VAD.from_hparams(
    source="speechbrain/vad-crdnn-libriparty",
    savedir="pretrained_models/vad-crdnn"
)

print(vad.mods)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


hyperparams.yaml: 0.00B [00:00, ?B/s]

model.ckpt:   0%|          | 0.00/453k [00:00<?, ?B/s]

mean_var_norm.ckpt:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

ModuleDict(
  (compute_features): Fbank(
    (compute_STFT): STFT()
    (compute_fbanks): Filterbank()
    (compute_deltas): Deltas()
    (context_window): ContextWindow()
  )
  (model): ModuleList(
    (0): Sequential(
      (norm1): LayerNorm(
        (norm): LayerNorm((40,), eps=1e-05, elementwise_affine=True)
      )
      (cnn1): CNN_Block(
        (conv_1): Conv2d(
          (conv): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1))
        )
        (norm_1): LayerNorm(
          (norm): LayerNorm((40, 16), eps=1e-05, elementwise_affine=True)
        )
        (act_1): LeakyReLU(negative_slope=0.01)
        (conv_2): Conv2d(
          (conv): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1))
        )
        (norm_2): LayerNorm(
          (norm): LayerNorm((40, 16), eps=1e-05, elementwise_affine=True)
        )
        (act_2): LeakyReLU(negative_slope=0.01)
        (pooling): Pooling1d(
          (pool_layer): MaxPool2d(kernel_size=(1, 2), stride=(1, 2), padding=(0, 0), dilation

In [4]:
# Cell 4：Number of nodes before pruning
model = vad.mods['model'][2]  # DNN
total_params = 0
print("=== DNN Layer Structure ===")
for name, module in model.named_modules():
    if hasattr(module, 'weight') and len(module.weight.shape) == 2:
        params = module.weight.numel()
        if module.bias is not None:
            params += module.bias.numel()
        total_params += params
        print(f"{name}: weight{list(module.weight.shape)}, params={params}")

print(f"\nTotal number of parameters: {total_params}")

=== DNN Layer Structure ===
dnn1.linear.w: weight[16, 64], params=1040
dnn2.linear.w: weight[16, 16], params=272
lin.w: weight[1, 16], params=16

Total number of parameters: 1328


In [9]:
# Cell 5 : pruning function with BatchNorm sync
import torch
import torch.nn as nn

def prune_linear_and_bn(linear, bn, next_linear, pruning_ratio=0.3):
    """
    Prune neurons from linear layer, sync BatchNorm and next linear layer
    linear: current Linear layer
    bn: BatchNorm1d following the linear layer
    next_linear: next Linear layer whose input must be updated
    """
    weight = linear.weight.data  # [out, in]

    # compute importance score per neuron (L1 norm)
    importance = torch.norm(weight, p=1, dim=1)

    # decide how many neurons to keep
    n_total = len(importance)
    n_keep = max(1, int(n_total * (1 - pruning_ratio)))
    print(f"  neurons: {n_total} -> {n_keep} (pruned {n_total - n_keep})")

    # find indices of most important neurons
    keep_idx = torch.argsort(importance, descending=True)[:n_keep]
    keep_idx = torch.sort(keep_idx).values

    # build new linear layer
    new_linear = nn.Linear(linear.in_features, n_keep, bias=linear.bias is not None)
    new_linear.weight.data = weight[keep_idx]
    if linear.bias is not None:
        new_linear.bias.data = linear.bias.data[keep_idx]

    # build new BatchNorm layer with matching size
    new_bn = nn.BatchNorm1d(n_keep, eps=bn.eps, momentum=bn.momentum, affine=bn.affine)
    new_bn.weight.data    = bn.weight.data[keep_idx]
    new_bn.bias.data      = bn.bias.data[keep_idx]
    new_bn.running_mean   = bn.running_mean[keep_idx]
    new_bn.running_var    = bn.running_var[keep_idx]

    # fix next layer's input dimension
    new_next = nn.Linear(n_keep, next_linear.out_features, bias=next_linear.bias is not None)
    new_next.weight.data = next_linear.weight.data[:, keep_idx]
    if next_linear.bias is not None:
        new_next.bias.data = next_linear.bias.data

    return new_linear, new_bn, new_next


def prune_vad_dnn(vad_model, pruning_ratio=0.3):
    dnn = vad_model.mods['model'][2]

    dnn1_linear = dnn.dnn1.linear.w
    dnn1_bn     = dnn.dnn1.norm.norm
    dnn2_linear = dnn.dnn2.linear.w
    dnn2_bn     = dnn.dnn2.norm.norm
    lin_linear  = dnn.lin.w

    print(f"=== Pruning DNN (ratio={pruning_ratio}) ===")

    # prune dnn1, sync dnn1 BN and dnn2 input
    print("Pruning dnn1:")
    new_dnn1, new_bn1, new_dnn2 = prune_linear_and_bn(dnn1_linear, dnn1_bn, dnn2_linear, pruning_ratio)
    dnn.dnn1.linear.w  = new_dnn1
    dnn.dnn1.norm.norm = new_bn1

    # prune dnn2, sync dnn2 BN and lin input
    print("Pruning dnn2:")
    new_dnn2, new_bn2, new_lin = prune_linear_and_bn(new_dnn2, dnn2_bn, lin_linear, pruning_ratio)
    dnn.dnn2.linear.w  = new_dnn2
    dnn.dnn2.norm.norm = new_bn2
    dnn.lin.w          = new_lin

    print("=== Pruning complete ===")

# reload the original model first, then prune
from speechbrain.inference.VAD import VAD
vad = VAD.from_hparams(
    source="speechbrain/vad-crdnn-libriparty",
    savedir="pretrained_models/vad-crdnn"
)
prune_vad_dnn(vad, pruning_ratio=0.3)

=== Pruning DNN (ratio=0.3) ===
Pruning dnn1:
  neurons: 16 -> 11 (pruned 5)
Pruning dnn2:
  neurons: 16 -> 11 (pruned 5)
=== Pruning complete ===


In [10]:
# Cell 6：Verify the number of parameters after pruning
model = vad.mods['model'][2]

total_params = 0
print("=== DNN layer structure after pruning ===")
for name, module in model.named_modules():
    if hasattr(module, 'weight') and len(module.weight.shape) == 2:
        params = module.numel() if not hasattr(module, 'bias') else module.weight.numel()
        if hasattr(module, 'bias') and module.bias is not None:
            params += module.bias.numel()
        total_params += params
        print(f"{name}: weight{list(module.weight.shape)}, params={params}")

print(f"\nTotal number of parameters: {total_params}")
print(f"Compression ratio: {total_params/1328:.2%}")

=== 剪枝后 DNN 层结构 ===
dnn1.linear.w: weight[11, 64], params=715
dnn2.linear.w: weight[11, 11], params=132
lin.w: weight[1, 11], params=11

总参数量: 858
压缩比: 64.61%


In [11]:
# Cell 7: verify pruned DNN with correct input dimension
import torch

dummy_input = torch.randn(1, 100, 64)

try:
    with torch.no_grad():
        output = vad.mods['model'][2](dummy_input)
    print("Forward pass successful!")
    print(f"Input shape:  {dummy_input.shape}")
    print(f"Output shape: {output.shape}")
except Exception as e:
    print(f"Error: {e}")

Forward pass successful!
Input shape:  torch.Size([1, 100, 64])
Output shape: torch.Size([1, 100, 1])


In [13]:
# Cell 8a: find the example audio file
import os

for root, dirs, files in os.walk("pretrained_models"):
    for f in files:
        print(os.path.join(root, f))

pretrained_models/vad-crdnn/hyperparams.yaml
pretrained_models/vad-crdnn/mean_var_norm.ckpt
pretrained_models/vad-crdnn/model.ckpt


In [14]:
# Cell 8b: download example audio from HuggingFace
from huggingface_hub import hf_hub_download

audio_path = hf_hub_download(
    repo_id="speechbrain/vad-crdnn-libriparty",
    filename="example_vad.wav",
    local_dir="pretrained_models/vad-crdnn"
)
print(f"Downloaded to: {audio_path}")

example_vad.wav:   0%|          | 0.00/9.59M [00:00<?, ?B/s]

Downloaded to: pretrained_models/vad-crdnn/example_vad.wav


In [15]:
# Cell 8: verify full pipeline with real audio
example_audio = "pretrained_models/vad-crdnn/example_vad.wav"

try:
    boundaries = vad.get_speech_segments(example_audio)
    vad.save_boundaries(boundaries)
    print("Full pipeline works!")
    print(boundaries)
except Exception as e:
    print(f"Error: {e}")

segment_001  0.00  5.96 SPEECH
segment_002  5.96  14.08 NON_SPEECH
segment_003  14.08  17.41 SPEECH
segment_004  17.41  18.10 NON_SPEECH
segment_005  18.10  22.28 SPEECH
segment_006  22.28  28.52 NON_SPEECH
segment_007  28.52  37.10 SPEECH
segment_008  37.10  37.69 NON_SPEECH
segment_009  37.69  51.56 SPEECH
segment_010  51.56  62.52 NON_SPEECH
segment_011  62.52  74.50 SPEECH
segment_012  74.50  80.58 NON_SPEECH
segment_013  80.58  95.64 SPEECH
segment_014  95.64  97.02 NON_SPEECH
segment_015  97.02  103.60 SPEECH
segment_016  103.60  110.00 NON_SPEECH
segment_017  110.00  110.64 SPEECH
segment_018  110.64  119.34 NON_SPEECH
segment_019  119.34  120.25 SPEECH
segment_020  120.25  121.58 NON_SPEECH
segment_021  121.58  131.08 SPEECH
segment_022  131.08  147.11 NON_SPEECH
segment_023  147.11  164.96 SPEECH
segment_024  164.96  167.41 NON_SPEECH
segment_025  167.41  172.66 SPEECH
segment_026  172.66  180.00 NON_SPEECH
segment_027  180.00  181.46 SPEECH
segment_028  181.46  186.28 NON_SPE

In [16]:
# Cell 9: reload original model and run baseline
from speechbrain.inference.VAD import VAD

vad_original = VAD.from_hparams(
    source="speechbrain/vad-crdnn-libriparty",
    savedir="pretrained_models/vad-crdnn"
)

example_audio = "pretrained_models/vad-crdnn/example_vad.wav"
boundaries_original = vad_original.get_speech_segments(example_audio)

print("=== Original model ===")
vad_original.save_boundaries(boundaries_original)

=== Original model ===
segment_001  0.00  5.60 SPEECH
segment_002  5.60  14.34 NON_SPEECH
segment_003  14.34  17.32 SPEECH
segment_004  17.32  18.15 NON_SPEECH
segment_005  18.15  21.63 SPEECH
segment_006  21.63  28.56 NON_SPEECH
segment_007  28.56  36.90 SPEECH
segment_008  36.90  38.07 NON_SPEECH
segment_009  38.07  51.26 SPEECH
segment_010  51.26  62.54 NON_SPEECH
segment_011  62.54  74.42 SPEECH
segment_012  74.42  80.99 NON_SPEECH
segment_013  80.99  94.65 SPEECH
segment_014  94.65  97.20 NON_SPEECH
segment_015  97.20  103.13 SPEECH
segment_016  103.13  121.99 NON_SPEECH
segment_017  121.99  128.80 SPEECH
segment_018  128.80  147.61 NON_SPEECH
segment_019  147.61  164.60 SPEECH
segment_020  164.60  167.47 NON_SPEECH
segment_021  167.47  172.65 SPEECH
segment_022  172.65  205.78 NON_SPEECH
segment_023  205.78  230.54 SPEECH
segment_024  230.54  237.61 NON_SPEECH
segment_025  237.61  245.98 SPEECH
segment_026  245.98  246.73 NON_SPEECH
segment_027  246.73  257.26 SPEECH
segment_028 

In [17]:
# Cell 10: compare parameter count before and after pruning
def count_dnn_params(vad_model):
    total = 0
    for name, module in vad_model.mods['model'][2].named_modules():
        if hasattr(module, 'weight') and len(module.weight.shape) == 2:
            total += module.weight.numel()
            if module.bias is not None:
                total += module.bias.numel()
    return total

params_original = count_dnn_params(vad_original)
params_pruned   = count_dnn_params(vad)

print(f"Original params: {params_original}")
print(f"Pruned params:   {params_pruned}")
print(f"Reduction:       {(1 - params_pruned/params_original):.1%}")
print(f"Pruned segments: {len(boundaries_original)}")
print(f"Original segs:   {len(boundaries_original)}")

Original params: 1328
Pruned params:   858
Reduction:       35.4%
Pruned segments: 15
Original segs:   15


In [20]:
# Cell 11: inference speed comparison
import time

example_audio = "pretrained_models/vad-crdnn/example_vad.wav"
N = 3

# original model speed
times_original = []
for i in range(N):
    print(f"Original run {i+1}/{N}...")
    start = time.time()
    vad_original.get_speech_segments(example_audio)
    times_original.append(time.time() - start)
    print(f"  done: {times_original[-1]:.3f}s")

# pruned model speed
times_pruned = []
for i in range(N):
    print(f"Pruned run {i+1}/{N}...")
    start = time.time()
    vad.get_speech_segments(example_audio)
    times_pruned.append(time.time() - start)
    print(f"  done: {times_pruned[-1]:.3f}s")

avg_original = sum(times_original) / N
avg_pruned   = sum(times_pruned) / N

print(f"\nOriginal model avg time: {avg_original:.3f}s")
print(f"Pruned model avg time:   {avg_pruned:.3f}s")
print(f"Speedup: {avg_original/avg_pruned:.2f}x")

Original run 1/3...
  done: 8.539s
Original run 2/3...
  done: 8.890s
Original run 3/3...
  done: 8.371s
Pruned run 1/3...
  done: 9.221s
Pruned run 2/3...
  done: 9.452s
Pruned run 3/3...
  done: 9.287s

Original model avg time: 8.600s
Pruned model avg time:   9.320s
Speedup: 0.92x


In [21]:
# Cell 12: check available disk space
import shutil

total, used, free = shutil.disk_usage("/")
print(f"Total: {total // (2**30)} GB")
print(f"Used:  {used  // (2**30)} GB")
print(f"Free:  {free  // (2**30)} GB")

Total: 107 GB
Used:  22 GB
Free:  85 GB


In [23]:
# Cell 13b: download LibriParty from OpenSLR
import os
os.makedirs("data", exist_ok=True)

!wget -q --show-progress "https://www.openslr.org/resources/107/LibriParty.tar.gz" -O data/LibriParty.tar.gz
print("Download complete!")

Download complete!


In [26]:
# Cell 13c: check if file was downloaded correctly
import os
size = os.path.getsize("data/LibriParty.tar.gz")
print(f"File size: {size} bytes")

File size: 0 bytes


In [27]:
# Cell 13d: download LibriSpeech dev-clean instead
!wget -q --show-progress "https://www.openslr.org/resources/12/dev-clean.tar.gz" -O data/dev-clean.tar.gz
print("Download complete!")

data/dev-clean.tar. 100%[===================>] 322.27M  13.8MB/s    in 24s     
Download complete!


In [30]:
# Cell 14: extract LibriSpeech dev-clean
import tarfile
import os

print("Extracting...")
with tarfile.open("data/dev-clean.tar.gz", "r:gz") as tar:
    tar.extractall("data/")
print("Extraction complete!")

# check the structure
for root, dirs, files in os.walk("data/LibriSpeech"):
    # only show top 2 levels
    level = root.replace("data/LibriSpeech", "").count(os.sep)
    if level < 2:
        print(root)

Extracting...


/tmp/ipykernel_8113/1426686135.py:7: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall("data/")


Extraction complete!
data/LibriSpeech
data/LibriSpeech/dev-clean


In [31]:
# Cell 15: explore LibriSpeech structure
import os

for root, dirs, files in os.walk("data/LibriSpeech/dev-clean"):
    level = root.replace("data/LibriSpeech/dev-clean", "").count(os.sep)
    if level < 3:
        indent = "  " * level
        print(f"{indent}{os.path.basename(root)}/")
        if level == 2:
            # show a few files
            subindent = "  " * (level + 1)
            for f in files[:3]:
                print(f"{subindent}{f}")

dev-clean/
  8842/
    302203/
      8842-302203-0007.flac
      8842-302203-0002.flac
      8842-302203-0011.flac
    304647/
      8842-304647-0006.flac
      8842-304647-0003.flac
      8842-304647-0012.flac
    302196/
      8842-302196-0004.flac
      8842-302196-0000.flac
      8842-302196-0001.flac
    302201/
      8842-302201-0014.flac
      8842-302201-0008.flac
      8842-302201-0004.flac
  2035/
    147961/
      2035-147961-0024.flac
      2035-147961-0018.flac
      2035-147961-0016.flac
    147960/
      2035-147960-0000.flac
      2035-147960.trans.txt
      2035-147960-0006.flac
    152373/
      2035-152373-0003.flac
      2035-152373-0004.flac
      2035-152373-0010.flac
  2086/
    149214/
      2086-149214-0000.flac
      2086-149214-0004.flac
      2086-149214-0002.flac
    149220/
      2086-149220-0033.flac
      2086-149220-0012.flac
      2086-149220-0004.flac
  1993/
    147964/
      1993-147964-0003.flac
      1993-147964-0000.flac
      1993-147964-0001.fl

In [32]:
# Cell 16: collect all flac files
import os
import glob

flac_files = glob.glob("data/LibriSpeech/dev-clean/**/*.flac", recursive=True)
print(f"Total audio files: {len(flac_files)}")
print("Example:", flac_files[0])

Total audio files: 2703
Example: data/LibriSpeech/dev-clean/8842/302203/8842-302203-0007.flac


In [33]:
# Cell 17: prepare training data (speech + silence pairs)
import torch
import torchaudio
import random

random.seed(42)

def load_and_resample(path, target_sr=16000):
    wav, sr = torchaudio.load(path)
    if sr != target_sr:
        wav = torchaudio.functional.resample(wav, sr, target_sr)
    # convert to mono
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
    return wav

def make_silence(length, sr=16000):
    # generate gaussian noise at very low energy to simulate silence
    return torch.randn(1, length) * 0.001

# use a small subset for fine-tuning (50 files to keep it fast)
subset = random.sample(flac_files, 50)

speech_wavs  = []
silence_wavs = []

print("Loading audio files...")
for i, path in enumerate(subset):
    wav = load_and_resample(path)
    speech_wavs.append(wav)
    # create a silence segment of the same length
    silence_wavs.append(make_silence(wav.shape[1]))
    if (i+1) % 10 == 0:
        print(f"  {i+1}/50 loaded")

print(f"Speech samples:  {len(speech_wavs)}")
print(f"Silence samples: {len(silence_wavs)}")
print(f"Example shape:   {speech_wavs[0].shape}")

Loading audio files...
  10/50 loaded
  20/50 loaded
  30/50 loaded
  40/50 loaded
  50/50 loaded
Speech samples:  50
Silence samples: 50
Example shape:   torch.Size([1, 245280])


In [34]:
# Cell 18: extract features using the VAD model's feature extractor
import torch

def extract_features(wav, vad_model):
    # use the model's built-in feature extractor (Fbank)
    with torch.no_grad():
        feats = vad_model.mods.compute_features(wav)
        feats = vad_model.mods.mean_var_norm(feats, torch.ones(1))
    return feats

# test feature extraction
sample_feat = extract_features(speech_wavs[0], vad)
print(f"Input wav shape:  {speech_wavs[0].shape}")
print(f"Output feat shape: {sample_feat.shape}")

Input wav shape:  torch.Size([1, 245280])
Output feat shape: torch.Size([1, 1534, 40])


In [38]:
# Cell 19: fine-tune the pruned DNN
import torch
import torch.nn as nn

optimizer = torch.optim.Adam(vad.mods['model'][2].parameters(), lr=1e-4)
criterion = nn.BCEWithLogitsLoss()

vad.mods['model'].train()

n_epochs = 5

for epoch in range(n_epochs):
    total_loss = 0
    correct = 0
    total = 0

    indices = list(range(50))
    random.shuffle(indices)

    for i in indices:
        # --- speech sample (label = 1) ---
        feat_s = extract_features(speech_wavs[i], vad)

        with torch.no_grad():
            cnn_out = vad.mods['model'][0](feat_s)
            cnn_out = cnn_out.reshape(cnn_out.shape[0], cnn_out.shape[1], -1)
            rnn_out = vad.mods['model'][1](cnn_out)
            if isinstance(rnn_out, tuple):
                rnn_out = rnn_out[0]

        optimizer.zero_grad()
        dnn_out = vad.mods['model'][2](rnn_out)
        label_s = torch.ones_like(dnn_out)
        loss_s  = criterion(dnn_out, label_s)

        # --- silence sample (label = 0) ---
        feat_n = extract_features(silence_wavs[i], vad)

        with torch.no_grad():
            cnn_out_n = vad.mods['model'][0](feat_n)
            cnn_out_n = cnn_out_n.reshape(cnn_out_n.shape[0], cnn_out_n.shape[1], -1)
            rnn_out_n = vad.mods['model'][1](cnn_out_n)
            if isinstance(rnn_out_n, tuple):
                rnn_out_n = rnn_out_n[0]

        dnn_out_n = vad.mods['model'][2](rnn_out_n)
        label_n   = torch.zeros_like(dnn_out_n)
        loss_n    = criterion(dnn_out_n, label_n)

        loss = loss_s + loss_n
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        pred_s = (torch.sigmoid(dnn_out) > 0.5).float()
        pred_n = (torch.sigmoid(dnn_out_n) > 0.5).float()
        correct += pred_s.sum().item() + (1 - pred_n).sum().item()
        total   += pred_s.numel() + pred_n.numel()

    avg_loss = total_loss / 50
    accuracy = correct / total * 100
    print(f"Epoch {epoch+1}/{n_epochs} | Loss: {avg_loss:.4f} | Accuracy: {accuracy:.2f}%")

print("Fine-tuning complete!")

Epoch 1/5 | Loss: 4.8894 | Accuracy: 52.68%
Epoch 2/5 | Loss: 4.7281 | Accuracy: 53.42%
Epoch 3/5 | Loss: 4.4889 | Accuracy: 53.24%
Epoch 4/5 | Loss: 4.2290 | Accuracy: 53.24%
Epoch 5/5 | Loss: 4.1131 | Accuracy: 52.79%
Fine-tuning complete!


In [39]:
# Cell 20: continue fine-tuning for more epochs
n_epochs_more = 15

for epoch in range(n_epochs_more):
    total_loss = 0
    correct = 0
    total = 0

    indices = list(range(50))
    random.shuffle(indices)

    for i in indices:
        feat_s = extract_features(speech_wavs[i], vad)

        with torch.no_grad():
            cnn_out = vad.mods['model'][0](feat_s)
            cnn_out = cnn_out.reshape(cnn_out.shape[0], cnn_out.shape[1], -1)
            rnn_out = vad.mods['model'][1](cnn_out)
            if isinstance(rnn_out, tuple):
                rnn_out = rnn_out[0]

        optimizer.zero_grad()
        dnn_out = vad.mods['model'][2](rnn_out)
        label_s = torch.ones_like(dnn_out)
        loss_s  = criterion(dnn_out, label_s)

        feat_n = extract_features(silence_wavs[i], vad)

        with torch.no_grad():
            cnn_out_n = vad.mods['model'][0](feat_n)
            cnn_out_n = cnn_out_n.reshape(cnn_out_n.shape[0], cnn_out_n.shape[1], -1)
            rnn_out_n = vad.mods['model'][1](cnn_out_n)
            if isinstance(rnn_out_n, tuple):
                rnn_out_n = rnn_out_n[0]

        dnn_out_n = vad.mods['model'][2](rnn_out_n)
        label_n   = torch.zeros_like(dnn_out_n)
        loss_n    = criterion(dnn_out_n, label_n)

        loss = loss_s + loss_n
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        pred_s = (torch.sigmoid(dnn_out) > 0.5).float()
        pred_n = (torch.sigmoid(dnn_out_n) > 0.5).float()
        correct += pred_s.sum().item() + (1 - pred_n).sum().item()
        total   += pred_s.numel() + pred_n.numel()

    avg_loss = total_loss / 50
    accuracy = correct / total * 100
    print(f"Epoch {epoch+6}/{n_epochs_more+5} | Loss: {avg_loss:.4f} | Accuracy: {accuracy:.2f}%")

print("Done!")

Epoch 6/20 | Loss: 3.9605 | Accuracy: 53.46%
Epoch 7/20 | Loss: 3.8216 | Accuracy: 53.43%
Epoch 8/20 | Loss: 3.6224 | Accuracy: 53.75%
Epoch 9/20 | Loss: 3.4833 | Accuracy: 53.75%
Epoch 10/20 | Loss: 3.2943 | Accuracy: 54.05%
Epoch 11/20 | Loss: 3.1592 | Accuracy: 53.60%
Epoch 12/20 | Loss: 2.9782 | Accuracy: 54.17%
Epoch 13/20 | Loss: 2.8416 | Accuracy: 52.67%
Epoch 14/20 | Loss: 2.7260 | Accuracy: 52.86%
Epoch 15/20 | Loss: 2.5404 | Accuracy: 53.95%
Epoch 16/20 | Loss: 2.3920 | Accuracy: 53.06%
Epoch 17/20 | Loss: 2.2487 | Accuracy: 54.26%
Epoch 18/20 | Loss: 2.1647 | Accuracy: 54.20%
Epoch 19/20 | Loss: 2.0273 | Accuracy: 54.66%
Epoch 20/20 | Loss: 1.8726 | Accuracy: 57.45%
Done!


In [40]:
# Cell 21: continue fine-tuning for 20 more epochs
n_epochs_more = 20

for epoch in range(n_epochs_more):
    total_loss = 0
    correct = 0
    total = 0

    indices = list(range(50))
    random.shuffle(indices)

    for i in indices:
        feat_s = extract_features(speech_wavs[i], vad)

        with torch.no_grad():
            cnn_out = vad.mods['model'][0](feat_s)
            cnn_out = cnn_out.reshape(cnn_out.shape[0], cnn_out.shape[1], -1)
            rnn_out = vad.mods['model'][1](cnn_out)
            if isinstance(rnn_out, tuple):
                rnn_out = rnn_out[0]

        optimizer.zero_grad()
        dnn_out = vad.mods['model'][2](rnn_out)
        label_s = torch.ones_like(dnn_out)
        loss_s  = criterion(dnn_out, label_s)

        feat_n = extract_features(silence_wavs[i], vad)

        with torch.no_grad():
            cnn_out_n = vad.mods['model'][0](feat_n)
            cnn_out_n = cnn_out_n.reshape(cnn_out_n.shape[0], cnn_out_n.shape[1], -1)
            rnn_out_n = vad.mods['model'][1](cnn_out_n)
            if isinstance(rnn_out_n, tuple):
                rnn_out_n = rnn_out_n[0]

        dnn_out_n = vad.mods['model'][2](rnn_out_n)
        label_n   = torch.zeros_like(dnn_out_n)
        loss_n    = criterion(dnn_out_n, label_n)

        loss = loss_s + loss_n
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        pred_s = (torch.sigmoid(dnn_out) > 0.5).float()
        pred_n = (torch.sigmoid(dnn_out_n) > 0.5).float()
        correct += pred_s.sum().item() + (1 - pred_n).sum().item()
        total   += pred_s.numel() + pred_n.numel()

    avg_loss = total_loss / 50
    accuracy = correct / total * 100
    print(f"Epoch {epoch+21}/{n_epochs_more+20} | Loss: {avg_loss:.4f} | Accuracy: {accuracy:.2f}%")

print("Done!")

Epoch 21/40 | Loss: 1.8326 | Accuracy: 55.09%
Epoch 22/40 | Loss: 1.7205 | Accuracy: 59.03%
Epoch 23/40 | Loss: 1.6449 | Accuracy: 59.39%
Epoch 24/40 | Loss: 1.5825 | Accuracy: 60.03%
Epoch 25/40 | Loss: 1.5182 | Accuracy: 61.37%
Epoch 26/40 | Loss: 1.4343 | Accuracy: 63.70%
Epoch 27/40 | Loss: 1.3760 | Accuracy: 65.46%
Epoch 28/40 | Loss: 1.3462 | Accuracy: 65.15%
Epoch 29/40 | Loss: 1.3186 | Accuracy: 66.49%
Epoch 30/40 | Loss: 1.2529 | Accuracy: 68.69%
Epoch 31/40 | Loss: 1.2054 | Accuracy: 68.47%
Epoch 32/40 | Loss: 1.2031 | Accuracy: 70.04%
Epoch 33/40 | Loss: 1.1555 | Accuracy: 71.15%
Epoch 34/40 | Loss: 1.1374 | Accuracy: 71.66%
Epoch 35/40 | Loss: 1.1188 | Accuracy: 72.57%
Epoch 36/40 | Loss: 1.0867 | Accuracy: 73.55%
Epoch 37/40 | Loss: 1.0674 | Accuracy: 74.43%
Epoch 38/40 | Loss: 1.0674 | Accuracy: 74.24%
Epoch 39/40 | Loss: 1.0529 | Accuracy: 75.26%
Epoch 40/40 | Loss: 1.0249 | Accuracy: 76.28%
Done!


In [41]:
# Cell 22: compare total model size before and after pruning
import torch

def count_total_params(vad_model):
    total = 0
    for p in vad_model.mods.parameters():
        total += p.numel()
    return total

def get_model_size_mb(vad_model):
    torch.save(vad_model.mods.state_dict(), "/tmp/model_temp.pt")
    import os
    size = os.path.getsize("/tmp/model_temp.pt") / (1024 ** 2)
    return size

params_pruned = count_total_params(vad)
size_pruned   = get_model_size_mb(vad)

params_original = count_total_params(vad_original)
size_original   = get_model_size_mb(vad_original)

print(f"=== Total model comparison ===")
print(f"Original  - params: {params_original:,}, size: {size_original:.2f} MB")
print(f"Pruned    - params: {params_pruned:,}, size: {size_pruned:.2f} MB")
print(f"Param reduction: {(1 - params_pruned/params_original):.1%}")
print(f"Size reduction:  {(1 - size_pruned/size_original):.1%}")

=== Total model comparison ===
Original  - params: 109,744, size: 0.44 MB
Pruned    - params: 109,254, size: 0.44 MB
Param reduction: 0.4%
Size reduction:  0.6%
